# Week 2.3 — SOAR Correlations, Thresholding, and Tapering

This notebook is **Problem 10**, worked. As before: run the cells and check the output against the
"What you should see" notes; the *Optional* section is not required.

Data assimilation needs a spatial covariance matrix. The honest ones are dense, and dense is
unaffordable at operational sizes, so the temptation is to set the small entries to zero. This
notebook is about what that costs — and it is not what most people guess.

1. the SOAR correlation matrix on a ring, and why it is circulant;
2. how badly conditioned it gets as the correlation length and the resolution grow;
3. thresholding: sparsity bought at the price of positive definiteness;
4. tapering: the same sparsity, definiteness kept.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Part 1: the SOAR matrix

$N$ equally spaced points on a circle of radius $a$. Between points $i$ and $j$ the angle is
$\theta_{ij}=2\pi(i-j)/N$, and the straight-line (chord) distance between them is
$r_{ij}=|2a\sin(\theta_{ij}/2)|$. The second-order autoregressive correlation function is

$$C(i,j)=\Bigl(1+\frac{r_{ij}}{L}\Bigr)\exp\Bigl(-\frac{r_{ij}}{L}\Bigr),$$

with $L>0$ the correlation lengthscale. Take $a=1$ throughout.

Because $r_{ij}$ depends only on $i-j$ modulo $N$, every row is the previous one shifted by one:
$C$ is **circulant**, so its eigenvalues are the discrete Fourier transform of its first row. That
is worth checking rather than assuming.

In [ ]:
def soar(N, L, a=1.0):
    # SOAR correlation matrix for N equally spaced points on a circle of radius a.
    k = np.arange(N)
    theta = 2 * np.pi * np.subtract.outer(k, k) / N
    r = np.abs(2 * a * np.sin(theta / 2))
    return (1 + r / L) * np.exp(-r / L)


C = soar(100, 0.4)

print(f"symmetric?           {np.allclose(C, C.T)}")
print(f"unit diagonal?       {np.allclose(np.diag(C), 1.0)}")
print(f"circulant?           {np.allclose(C[1], np.roll(C[0], 1))}")

# For a circulant matrix the eigenvalues are the DFT of the first row.
lam_fft = np.sort(np.real(np.fft.fft(C[0])))
lam_eig = np.sort(np.linalg.eigvalsh(C))
print(f"\nmax |FFT eigenvalues - eigh eigenvalues| = {np.max(np.abs(lam_fft - lam_eig)):.2e}")
print(f"smallest eigenvalue  {lam_eig[0]:.4e}   largest {lam_eig[-1]:.4f}")

**What you should see.** All three structural checks `True`, and the FFT eigenvalues agreeing with
`eigh` to round-off. The matrix is symmetric positive definite but only just: its smallest
eigenvalue is about $3\cdot10^{-4}$ against a largest of about $28$.

It is also completely dense — $10^4$ nonzeros for $100$ points. That is the problem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

axes[0].plot(C[0][:51], lw=2)
axes[0].axhline(0.2, color="C3", ls="--", lw=1.3, label=r"threshold $\tau=0.2$")
axes[0].set_xlabel("grid points from the diagonal")
axes[0].set_ylabel("correlation")
axes[0].set_title("First row of $C$  ($N=100$, $L=0.4$)")
axes[0].grid(True, alpha=0.25)
axes[0].legend(fontsize=9)

im = axes[1].imshow(C, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("$C$ — dense, banded-looking, but never zero")
fig.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

The left panel is the whole story in one picture: the correlation decays quickly but it has a
**long tail** that never reaches zero. Everything below the dashed line is what thresholding throws
away.

## Part 2: conditioning grows with the lengthscale and with the resolution

In [ ]:
print(f"{'N':>6} {'L':>6} {'lambda_min':>13} {'kappa_2(C)':>13}")
for N, L in [(100, 0.1), (100, 0.4), (200, 0.4)]:
    w = np.linalg.eigvalsh(soar(N, L))
    print(f"{N:>6} {L:>6} {w[0]:>13.2e} {w[-1]/w[0]:>13.2e}")

**What you should see.**

| $N$ | $L$ | $\lambda_{\min}$ | $\kappa_2(C)$ |
|---|---|---|---|
| 100 | 0.1 | $1.92\cdot10^{-2}$ | $3.34\cdot10^{2}$ |
| 100 | 0.4 | $3.22\cdot10^{-4}$ | $8.70\cdot10^{4}$ |
| 200 | 0.4 | $4.03\cdot10^{-5}$ | $1.39\cdot10^{6}$ |

Both knobs make it worse, for the same underlying reason. A longer $L$ makes neighbouring columns
more nearly parallel; refining the grid at fixed $L$ does exactly the same thing, because the new
points sit *between* old ones that were already strongly correlated. Neither is a numerical
artefact — it is the physics of a smooth correlation function.

## Part 3: thresholding buys sparsity and loses positive definiteness

The obvious move: set $C_{ij}=0$ whenever $C_{ij}<\tau$. The chapter mentions $\tau=0.2$ as a
typical operational choice. It certainly produces sparsity — the question is what else it produces.

(On a ring the natural notion of bandwidth is the *circular* one, $\min(|i-j|,\,N-|i-j|)$, which is
at most $N/2$.)

In [ ]:
N, L = 100, 0.4
C = soar(N, L)
k = np.arange(N)
dist = np.abs(np.subtract.outer(k, k))
dist = np.minimum(dist, N - dist)          # circular distance

print(f"{'tau':>6} {'nnz per row':>13} {'circ. half-bw':>15} {'lambda_min':>14} {'SPD?':>7}")
spectra = {}
for tau in [0.0, 0.05, 0.2, 0.5]:
    Ct = np.where(C < tau, 0.0, C)
    w = np.linalg.eigvalsh(Ct)
    spectra[tau] = w
    nnz = int((Ct[0] != 0).sum())
    hb = int(dist[Ct != 0].max())
    print(f"{tau:>6} {nnz:>13} {hb:>15} {w[0]:>+14.3e} {str(w[0] > 0):>7}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for tau, w in spectra.items():
    ax.plot(np.sort(w), lw=2, label=rf"$\tau={tau}$")

ax.axhline(0, color="0.3", lw=1.2)
ax.axhspan(min(w.min() for w in spectra.values()) * 1.15, 0, color="C3", alpha=0.07)
ax.text(0.97, 0.06, "not positive definite", color="C3", fontsize=9,
        ha="right", transform=ax.transAxes)

ax.set_xlabel("eigenvalue index (sorted)")
ax.set_ylabel("eigenvalue")
ax.set_title("Thresholding pushes eigenvalues below zero")
ax.set_ylim(-2.4, 4.0)
ax.grid(True, alpha=0.25)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**What you should see.**

| $\tau$ | nnz per row | circular half-bandwidth | $\lambda_{\min}$ |
|---|---|---|---|
| $0$ | 100 | 50 | $+3.22\cdot10^{-4}$ |
| $0.05$ | 79 | 39 | $-9.06\cdot10^{-2}$ |
| $0.2$ | 41 | 20 | $-5.01\cdot10^{-1}$ |
| $0.5$ | 21 | 10 | $-1.90$ |

The sparsity is real — at $\tau=0.2$ the matrix has gone from $100$ nonzeros per row to $41$. But
look at the sign. **The very first threshold destroys positive definiteness**, and $\tau=0.05$ is a
gentle one: it only zeroes correlations below $5\%$.

This is not a small numerical wobble. At $\tau=0.2$ the smallest eigenvalue is $-0.5$, against a
largest of about $23$. The consequences are immediate and practical:

- Cholesky fails outright — there is no $LL^\top$;
- CG has no reason to converge, and breaks down;
- in a variational assimilation scheme the cost function is no longer convex, so the minimisation
  problem you have written down is not the one you meant to write down.

The truncated matrix is still circulant, so this is easy to see analytically too: thresholding
changes the first row, and the eigenvalues are its Fourier transform. Chopping a smooth decaying
sequence introduces high-frequency content, and nothing keeps the resulting transform positive.

**The point is the sign, not the sparsity.**

## Part 4: tapering — the same sparsity, definiteness kept

The fix used operationally is not to threshold but to **taper**: replace $C$ by the entrywise
(Schur) product $C\circ T$, where $T$ is itself a positive-definite correlation function that is
*compactly supported* — exactly zero beyond some radius.

Why this works is a one-line theorem. The Schur product theorem says the entrywise product of two
positive semidefinite matrices is positive semidefinite. So $C\circ T$ inherits definiteness from
its two factors, while inheriting its zeros from $T$. Sparsity and definiteness at once.

The standard choice is the Gaspari–Cohn function, a piecewise-quintic approximation to a Gaussian
that vanishes beyond $2c$.

In [ ]:
def gaspari_cohn(r, c):
    # Gaspari-Cohn (1999) compactly supported correlation: zero for r > 2c.
    x = np.abs(r) / c
    out = np.zeros_like(x)

    m = x <= 1
    z = x[m]
    out[m] = 1 - (5 / 3) * z**2 + (5 / 8) * z**3 + (1 / 2) * z**4 - (1 / 4) * z**5

    m = (x > 1) & (x <= 2)
    z = x[m]
    out[m] = (4 - 5 * z + (5 / 3) * z**2 + (5 / 8) * z**3
              - (1 / 2) * z**4 + (1 / 12) * z**5 - 2 / (3 * z))
    return out


k = np.arange(N)
theta = 2 * np.pi * np.subtract.outer(k, k) / N
r = np.abs(2 * np.sin(theta / 2))          # a = 1, same chord distance as in soar()

print(f"{'method':>22} {'nnz per row':>13} {'lambda_min':>14} {'kappa_2':>12} {'SPD?':>7}")

w = np.linalg.eigvalsh(C)
print(f"{'C (no truncation)':>22} {N:>13} {w[0]:>+14.3e} {w[-1]/w[0]:>12.2e} {str(w[0] > 0):>7}")

Ct = np.where(C < 0.2, 0.0, C)
w = np.linalg.eigvalsh(Ct)
print(f"{'thresholded, tau=0.2':>22} {int((Ct[0] != 0).sum()):>13} {w[0]:>+14.3e} "
      f"{'---':>12} {str(w[0] > 0):>7}")

for c in [0.35, 0.5]:
    T = gaspari_cohn(r, c)
    # The premise of the Schur-product argument: the taper is itself positive definite.
    assert np.linalg.eigvalsh(T)[0] > 0, "taper is not PD"

    Ctap = C * T
    w = np.linalg.eigvalsh(Ctap)
    nnz = int((Ctap[0] != 0).sum())
    kap = f"{w[-1]/w[0]:12.2e}" if w[0] > 0 else f"{'---':>12}"
    print(f"{'tapered, c=' + str(c):>22} {nnz:>13} {w[0]:>+14.3e} {kap} {str(w[0] > 0):>7}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))

T = gaspari_cohn(r, 0.35)
axes[0].plot(C[0][:51], lw=2, label="$C$ (SOAR)")
axes[0].plot(T[0][:51], lw=2, label="$T$ (Gaspari-Cohn, $c=0.35$)")
axes[0].plot((C * T)[0][:51], lw=2, label=r"$C\circ T$ (tapered)")
axes[0].plot(np.where(C[0][:51] < 0.2, 0.0, C[0][:51]), lw=2, ls="--",
             label=r"thresholded, $\tau=0.2$")
axes[0].axhline(0, color="0.3", lw=1.0)
axes[0].set_xlabel("grid points from the diagonal")
axes[0].set_ylabel("correlation")
axes[0].set_title("First row: four ways")
axes[0].grid(True, alpha=0.25)
axes[0].legend(fontsize=8)

# Zoom on the BOTTOM of the spectrum: the largest eigenvalues are around 25 and would
# squash the only part that matters, which is whether the curves cross zero.
axes[1].plot(np.sort(np.linalg.eigvalsh(C)), lw=2, label="$C$")
axes[1].plot(np.sort(np.linalg.eigvalsh(np.where(C < 0.2, 0.0, C))), lw=2, ls="--",
             label=r"thresholded, $\tau=0.2$")
axes[1].plot(np.sort(np.linalg.eigvalsh(C * T)), lw=2, label=r"tapered, $c=0.35$")
axes[1].axhline(0, color="0.3", lw=1.2)
axes[1].axhspan(-0.75, 0, color="C3", alpha=0.07)
axes[1].set_ylim(-0.75, 1.2)
axes[1].set_xlabel("eigenvalue index (sorted)")
axes[1].set_ylabel("eigenvalue")
axes[1].set_title("Bottom of the spectrum (the top is off-scale)")
axes[1].grid(True, alpha=0.25)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

**What you should see.** The tapered matrix at $c=0.35$ is **sparser** than the thresholded one —
$23$ nonzeros per row against $41$ — its smallest eigenvalue is **positive**, and the condition
number is not merely survivable but *better* than the original: $5.7\cdot10^{3}$ against
$8.7\cdot10^{4}$. Tapering is not a compromise that costs you conditioning to buy definiteness; here
it improves both at once.

In the left panel you can see why. Thresholding drops off a cliff at $\tau=0.2$; the taper eases
the correlation smoothly to exactly zero. That smoothness is what keeps the Fourier transform
positive, and hence the matrix definite. (The `assert` in the cell above checks the premise of the
Schur argument — the taper $T$ is itself positive definite, $\lambda_{\min}(T)\approx8.5\cdot10^{-4}$.)

The tapered matrix is not the same matrix as $C$ — it damps the near-diagonal correlations a little
as well, which is a real modelling cost. But it is a *usable* matrix: Cholesky succeeds, CG
converges, the variational cost function stays convex.

This is localisation in ensemble Kalman filtering, and it is the reason the chapter's Remark says
localisation *prescribes* a sparsity structure rather than merely truncating one.

## Optional

Not required.

- Verify the Schur product theorem numerically: generate random SPD $A$ and $B$ and check that
  $A\circ B$ is SPD. Then check that it fails for $A$ SPD and $B$ merely symmetric.
- Repeat Part 3 with $N=200$ and $N=400$ at fixed $L$. Does the most negative eigenvalue get worse
  with resolution? (It does. Why?)
- Find, for $L=0.4$ and $N=100$, the largest $\tau$ for which the thresholded matrix is still
  positive definite. Compare it with the smallest $C_{ij}$ in the matrix.
- The tapered matrix is banded and circulant. Time a Cholesky solve against a dense one, and
  against CG, at $N=2000$.